# Configurable Local-Evidence Cross-Dataset Evaluation

Evaluate a configurable matrix of local-evidence training and evaluation regimes.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display

from local_evidence_evaluation_utils import (
    combine_local_evidence_dataset_results,
    evaluate_local_evidence_datasets,
    evaluate_local_evidence_semeval,
)
from local_evidence_training_utils import train_local_evidence_models
from local_evidence_utils import (
    load_or_build_local_evidence_windows,
    split_local_evidence_texts,
)

sns.set_theme(style="whitegrid", context="notebook")
DATA_ROOT = Path(os.environ.get("TEXT_DETECTION_DATA", "D:/Projects/master_diploma/text-detection-data"))
PROCESSED_ROOT = DATA_ROOT / "processed"

In [ ]:
DATASETS = {"Kaggle": "kaggle_train", "Outfox": "outfox_chatgpt_train"}
LLM_MODELS = ["smollm2_360m"]
TRAINING_REGIMES = {
    "Kaggle": ("Kaggle",),
    "Outfox": ("Outfox",),
    "Mixed": ("Kaggle", "Outfox"),
}
EVALUATION_REGIMES = TRAINING_REGIMES
CLASSIFIER_NAMES = ("logistic_regression",)
WINDOW_LENGTHS = (32, 64, 128)
STRIDE = 16
TOP_K = 20
SEED = 42
RECOMPUTE_WINDOWS = False
SEMEVAL_DATASET = "semeval2024_task8_subtask_c_dev"
SEMEVAL_SPLIT = "MG"
SEMEVAL_PREPARED_PATH = DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_dev_MG.csv"
SEMEVAL_MG_SCORE_THRESHOLD = 0.5
SEMEVAL_SMOOTHING_WINDOW = 5
SEMEVAL_MINIMUM_REGION_LENGTH = 3

In [ ]:
metrics_frames = []
prediction_frames = []
scored_window_frames = []
coverage_frames = []
semeval_summary_frames = []
semeval_evaluation_frames = []
semeval_region_frames = []
semeval_scored_window_frames = []

for llm in LLM_MODELS:
    partitions_by_dataset = {}
    for dataset_name, dataset_alias in DATASETS.items():
        windows = load_or_build_local_evidence_windows(
            PROCESSED_ROOT, dataset_alias, llm, window_lengths=WINDOW_LENGTHS,
            stride=STRIDE, top_k=TOP_K, recompute=RECOMPUTE_WINDOWS,
        )
        partitions_by_dataset[dataset_name] = split_local_evidence_texts(windows, seed=SEED)
        coverage_frames.append(
            windows.groupby("split").size().rename("rows").reset_index().assign(
                llm=llm, source_dataset=dataset_name, training_regime="prepared",
                evaluation_regime="source"
            )
        )

    semeval_annotations = pd.read_csv(SEMEVAL_PREPARED_PATH)
    required_columns = {"id", "text", "boundary_word_index"}
    if required_columns.difference(semeval_annotations.columns):
        raise ValueError("Prepared SemEval CSV must contain id, text, and boundary_word_index columns.")

    for training_regime, training_names in TRAINING_REGIMES.items():
        training_aliases = tuple(DATASETS[name] for name in training_names)
        train_ids = {DATASETS[name]: partitions_by_dataset[name]["train"] for name in training_names}
        validation_ids = {DATASETS[name]: partitions_by_dataset[name]["validation"] for name in training_names}
        test_ids = {name: partitions_by_dataset[name]["test"] for name in DATASETS}
        trained = train_local_evidence_models(
            PROCESSED_ROOT, training_aliases, llm, train_ids, validation_ids,
            classifier_names=CLASSIFIER_NAMES, window_lengths=WINDOW_LENGTHS,
            stride=STRIDE, top_k=TOP_K, seed=SEED,
        )

        separate_results = []
        for evaluation_regime, evaluation_names in EVALUATION_REGIMES.items():
            if evaluation_regime == "Mixed":
                continue
            evaluation_name = evaluation_names[0]
            evaluated = evaluate_local_evidence_datasets(
                trained.model, PROCESSED_ROOT, (DATASETS[evaluation_name],),
                {DATASETS[evaluation_name]: test_ids[evaluation_name]},
            )
            for frame_name in ("metrics", "predictions", "scored_windows", "coverage"):
                frame = getattr(evaluated, frame_name).copy()
                frame["llm"] = llm
                frame["training_regime"] = training_regime
                frame["evaluation_regime"] = evaluation_regime
                frame["evaluation_dataset"] = DATASETS[evaluation_name]
                if frame_name == "metrics":
                    frame["classifier"] = frame["model"]
                    metrics_frames.append(frame)
                elif frame_name == "predictions":
                    prediction_frames.append(frame)
                elif frame_name == "scored_windows":
                    scored_window_frames.append(frame)
                else:
                    coverage_frames.append(frame)
            separate_results.append(evaluated)

        mixed = combine_local_evidence_dataset_results(separate_results, evaluation_regime="Mixed")
        for frame_name in ("metrics", "predictions", "scored_windows", "coverage"):
            frame = getattr(mixed, frame_name).copy()
            frame["llm"] = llm
            frame["training_regime"] = training_regime
            frame["evaluation_regime"] = "Mixed"
            frame["evaluation_dataset"] = "Mixed"
            if frame_name == "metrics":
                frame["classifier"] = frame["model"]
                metrics_frames.append(frame)
            elif frame_name == "predictions":
                prediction_frames.append(frame)
            elif frame_name == "scored_windows":
                scored_window_frames.append(frame)
            else:
                coverage_frames.append(frame)

        semeval = evaluate_local_evidence_semeval(
            trained.model, PROCESSED_ROOT, semeval_annotations, dataset=SEMEVAL_DATASET,
            split=SEMEVAL_SPLIT, threshold=SEMEVAL_MG_SCORE_THRESHOLD,
            smoothing_window=SEMEVAL_SMOOTHING_WINDOW,
            minimum_region_length=SEMEVAL_MINIMUM_REGION_LENGTH,
        )
        semeval_summary_frames.append(semeval.summary.assign(llm=llm, training_regime=training_regime, evaluation_regime="SemEval"))
        semeval_evaluation_frames.append(semeval.evaluations.assign(llm=llm, training_regime=training_regime, evaluation_regime="SemEval"))
        semeval_region_frames.append(semeval.predicted_regions.assign(llm=llm, training_regime=training_regime, evaluation_regime="SemEval"))
        semeval_scored_window_frames.append(semeval.scored_windows.assign(llm=llm, training_regime=training_regime, evaluation_regime="SemEval"))

metrics = pd.concat(metrics_frames, ignore_index=True)
predictions = pd.concat(prediction_frames, ignore_index=True)
scored_windows = pd.concat(scored_window_frames, ignore_index=True)
coverage = pd.concat(coverage_frames, ignore_index=True)
semeval_region_summary = pd.concat(semeval_summary_frames, ignore_index=True)
semeval_region_evaluations = pd.concat(semeval_evaluation_frames, ignore_index=True)
semeval_predicted_regions = pd.concat(semeval_region_frames, ignore_index=True)
semeval_scored_windows = pd.concat(semeval_scored_window_frames, ignore_index=True)

In [ ]:
display(coverage)
display(metrics)
display(semeval_region_summary)

for llm in LLM_MODELS:
    matrix = (
        metrics.query("llm == @llm and aggregation == 'evidence_fraction'")
        .pivot(index=["training_regime", "classifier", "window_length"], columns="evaluation_regime", values="macro_f1")
        .reindex(columns=list(EVALUATION_REGIMES))
    )
    display(matrix)
    plt.figure(figsize=(8, max(3, len(matrix) * 0.35)), constrained_layout=True)
    sns.heatmap(matrix, annot=True, fmt=".3f", vmin=0.0, vmax=1.0, cmap="viridis", mask=matrix.isna())
    plt.title(f"Local-evidence macro-F1 matrix: {llm}")
    plt.xlabel("Evaluation regime")
    plt.ylabel("Training regime / classifier / window length")
    plt.show()